# Quick Start: GDP Nowcast in 15 Lines

**Estimated time:** 2 minutes

This notebook demonstrates a complete one-step-ahead GDP nowcast using Beta MIDAS with three vintage points (1-month, 2-month, 3-month) and prints the expanding-window RMSE for each.

For the full nowcasting workflow, see Module 03.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist
import os, warnings
warnings.filterwarnings('ignore')

# --- Load data ---
def find_csv(filename):
    for base in ['../modules/module_00_foundations/resources',
                 'modules/module_00_foundations/resources']:
        path = os.path.join(base, filename)
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True).squeeze()
    raise FileNotFoundError(filename)

gdp_growth = find_csv('gdp_quarterly.csv')
ip_growth  = find_csv('industrial_production_monthly.csv')

# --- MIDAS matrix builder (ragged-edge aware) ---
def build_midas(y_low, x_high, K, h=0):
    y_q = y_low.copy(); y_q.index = y_low.index.to_period('Q')
    x_m = x_high.copy(); x_m.index = x_high.index.to_period('M')
    K_eff = K - h; rows_Y, rows_X = [], []
    for q in y_q.index:
        last = q.asfreq('M', how='end') - h
        lags = [last - i for i in range(K_eff)]
        if any(l not in x_m.index for l in lags): continue
        rows_Y.append(y_q[q]); rows_X.append([x_m[l] for l in lags])
    return np.array(rows_Y), np.array(rows_X)

def beta_w(K, t1, t2):
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, max(t1,0.01), max(t2,0.01))
    return raw / raw.sum() if raw.sum() > 1e-12 else np.ones(K)/K

def est_midas(Y, X):
    K_ = X.shape[1]
    def psse(theta):
        t1, t2 = theta
        if t1 <= 0.01 or t2 <= 0.01: return 1e10
        xw = X @ beta_w(K_, t1, t2)
        xc, yc = xw - xw.mean(), Y - Y.mean()
        ss = xc @ xc
        if ss < 1e-12: return np.sum((Y-Y.mean())**2)
        b = yc@xc/ss; a = Y.mean() - b*xw.mean()
        return np.sum((Y-a-b*xw)**2)
    best = min([minimize(psse, t0, method='Nelder-Mead',
                         options={'maxiter':20000,'xatol':1e-8})
                for t0 in [(1.,5.),(1.5,4.)]], key=lambda r: r.fun)
    t1, t2 = max(best.x[0],0.01), max(best.x[1],0.01)
    w_ = beta_w(K_, t1, t2); xw = X @ w_
    xc, yc = xw - xw.mean(), Y - Y.mean()
    b = yc@xc/(xc@xc); a = Y.mean() - b*xw.mean()
    return {'alpha': a, 'beta': b, 'theta1': t1, 'theta2': t2, 'weights': w_}

# --- Expanding-window RMSE for each vintage ---
K = 12; MIN_TRAIN = 30
print("Running expanding-window RMSE for 3 vintage points...")
print("(~1-2 minutes)")

rmse_by_h = {}
for h in [2, 1, 0]:  # 1-month, 2-month, 3-month
    Y_h, X_h = build_midas(gdp_growth, ip_growth, K, h=h)
    T = len(Y_h); sq_err = []
    for t in range(MIN_TRAIN, T):
        est = est_midas(Y_h[:t], X_h[:t])
        xw_t = float(X_h[t] @ beta_w(X_h.shape[1], est['theta1'], est['theta2']))
        y_hat = est['alpha'] + est['beta'] * xw_t
        sq_err.append((Y_h[t] - y_hat)**2)
    rmse = np.sqrt(np.mean(sq_err))
    rmse_by_h[h] = rmse
    vintage = {0: '3-month', 1: '2-month', 2: '1-month'}[h]
    print(f"  {vintage}: RMSE = {rmse:.4f}")

# --- AR(1) benchmark ---
Y_base, _ = build_midas(gdp_growth, ip_growth, K, h=0)
ar_errs = []
for t in range(MIN_TRAIN, len(Y_base)):
    Y_tr = Y_base[:t]
    Z = np.column_stack([np.ones(t-1), Y_tr[:-1]])
    p = np.linalg.lstsq(Z, Y_tr[1:], rcond=None)[0]
    ar_errs.append((Y_base[t] - (p[0] + p[1]*Y_base[t-1]))**2)
ar_rmse = np.sqrt(np.mean(ar_errs))
print(f"  AR(1) benchmark: RMSE = {ar_rmse:.4f}")

# --- Bar chart ---
models = ['AR(1)', '1-month', '2-month', '3-month']
rmses  = [ar_rmse, rmse_by_h[2], rmse_by_h[1], rmse_by_h[0]]
colors = ['#d62728', '#2ca02c', '#ff7f0e', '#1f77b4']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(models, rmses, color=colors, alpha=0.85)
for bar, rmse in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{rmse:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('OOS RMSE')
ax.set_title('GDP Nowcast RMSE by Model and Vintage')
ax.set_ylim(0, max(rmses)*1.3)
plt.tight_layout()
plt.show()
print("\nNowcast quick start complete. See Module 03 for full details.")